# QEC Examples

This notebook collects larger, demonstration-oriented QEC circuits for `lichen`. Unlike the validation notebook in `docs/tutorials/`, the focus here is on realistic circuit inputs and noisy-simulation workflow rather than analytic micro-checks.

The first section covers the Shor `[[9,1,3]]` encoding circuit under the current shared quasi-static model with fixed parameters

- `\Delta \equiv 1` for every raw noisy segment
- `\sigma = 0.2`
- `N_{MC} = 100` Monte Carlo shots


In [1]:
import os
import sys
import time
import runpy
from collections import Counter
from pathlib import Path

import numpy as np

def locate_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'src' / 'lichen').is_dir():
            return candidate
    raise RuntimeError('Could not locate the local src/lichen package')

REPO_ROOT = locate_repo_root()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lichen import (
    SharedQuasiStaticModel,
    build_prepared_blockwise_hidden_memory_process,
    prepare_blockwise_hidden_memory_simulation,
    sample_blockwise_hidden_memory_processes,
)

QEC_EXAMPLES_DIR = REPO_ROOT / 'examples' / 'QEC_examples'
SHOR_MODULE = runpy.run_path(str(QEC_EXAMPLES_DIR / 'Shor_9-1-3.py'))
build_shor_9_1_3_encoding_circuit = SHOR_MODULE['build_shor_9_1_3_encoding_circuit']
build_shor_9_1_3_layout = SHOR_MODULE['build_shor_9_1_3_layout']
plot_shor_9_1_3_encoding_circuit = SHOR_MODULE['plot_shor_9_1_3_encoding_circuit']

SURFACE_MODULE = runpy.run_path(str(QEC_EXAMPLES_DIR / 'Rotated_Surface_d2.py'))
build_rotated_surface_d2_syndrome_circuit = SURFACE_MODULE['build_rotated_surface_d2_syndrome_circuit']
build_rotated_surface_d2_layout = SURFACE_MODULE['build_rotated_surface_d2_layout']
plot_rotated_surface_d2_syndrome_circuit = SURFACE_MODULE['plot_rotated_surface_d2_syndrome_circuit']

SURFACE_D3_MODULE = runpy.run_path(str(QEC_EXAMPLES_DIR / 'Rotated_Surface_d3_memory_x.py'))
build_rotated_surface_d3_memory_x_circuit = SURFACE_D3_MODULE['build_rotated_surface_d3_memory_x_circuit']
build_rotated_surface_d3_memory_x_layout = SURFACE_D3_MODULE['build_rotated_surface_d3_memory_x_layout']
plot_rotated_surface_d3_memory_x_circuit = SURFACE_D3_MODULE['plot_rotated_surface_d3_memory_x_circuit']


## Shared Helpers

These helpers are generic to the notebook, so future QECC sections can reuse them without changing the import/setup cells.


In [2]:
WINDOW_SIZE = 2
DELTA = 1.0
SIGMA = 0.2
N_MC = 100
SEED = 0
NUM_WORKERS = min(4, os.cpu_count() or 1)


def run_mc_batch(model, circuit, *, window_size: int, num_shots: int, seed: int, num_workers: int):
    start = time.perf_counter()
    batch = sample_blockwise_hidden_memory_processes(
        model,
        circuit,
        window_size=window_size,
        num_shots=num_shots,
        rng=np.random.default_rng(seed),
        num_workers=num_workers,
    )
    elapsed = time.perf_counter() - start
    return batch, elapsed


def summarize_batch(batch) -> dict:
    xi_values = []
    nontrivial_counts = []
    physical_counters = None
    toggling_counters = None

    for process in batch.processes:
        xi_values.append(process.hidden_memory.xi)
        nontrivial_counts.append(sum(1 for fault in process.block_faults if fault.physical_pauli_string is not None))
        if physical_counters is None:
            physical_counters = [Counter() for _ in process.block_faults]
            toggling_counters = [Counter() for _ in process.block_faults]
        for block_index, fault in enumerate(process.block_faults):
            physical_label = fault.physical_pauli_string if fault.physical_pauli_string is not None else 'I'
            toggling_label = fault.toggling_frame_pauli_string if fault.toggling_frame_pauli_string is not None else 'I'
            physical_counters[block_index][physical_label] += 1
            toggling_counters[block_index][toggling_label] += 1

    xi_array = np.asarray(xi_values, dtype=float)
    return {
        'num_shots': len(batch.processes),
        'mean_xi': float(np.mean(xi_array)),
        'mean_xi2': float(np.mean(xi_array ** 2)),
        'std_xi': float(np.std(xi_array)),
        'avg_nontrivial_blocks': float(np.mean(nontrivial_counts)),
        'physical_frequencies': [
            {label: count / len(batch.processes) for label, count in counter.items()}
            for counter in physical_counters
        ],
        'toggling_frequencies': [
            {label: count / len(batch.processes) for label, count in counter.items()}
            for counter in toggling_counters
        ],
    }


def sort_probability_dict(probabilities: dict[str, float], *, top_k: int = 8) -> list[tuple[str, float]]:
    return sorted(probabilities.items(), key=lambda item: (-item[1], item[0]))[:top_k]


def print_frequency_table(title: str, frequencies: list[dict[str, float]]) -> None:
    print(title)
    for block_index, freq_dict in enumerate(frequencies):
        ordered = sorted(freq_dict.items(), key=lambda item: (-item[1], item[0]))
        print(f'  block {block_index}:', dict(ordered))


## Shor `[[9,1,3]]` Encoding Circuit

This section loads the Clifford encoding circuit from `Shor_9-1-3.py`, prints the named layout, and renders a lightweight ASCII circuit diagram. Later QEC examples can follow the same pattern in new sections below.


In [3]:
shor_layout = build_shor_9_1_3_layout()
shor_circuit = build_shor_9_1_3_encoding_circuit()

print('Shor [[9,1,3]] layout')
print('  total_qubits =', shor_layout.total_qubits)
print('  logical_data_qubit =', shor_layout.logical_data_qubit)
print('  block_leaders =', shor_layout.block_leaders)
print('  blocks =', shor_layout.blocks)
print('  circuit_depth =', shor_circuit.circuit_depth)
print('')
print('ASCII circuit:')
print(plot_shor_9_1_3_encoding_circuit())


Shor [[9,1,3]] layout
  total_qubits = 9
  logical_data_qubit = 0
  block_leaders = (0, 3, 6)
  blocks = ((0, 1, 2), (3, 4, 5), (6, 7, 8))
  circuit_depth = 11

ASCII circuit:
q0: --@------@------H--------------------@------@--------------------------------
q1: --|------|---------------------------X------|--------------------------------
q2: --|------|----------------------------------X--------------------------------
q3: --X------|-------------H---------------------------@------@------------------
q4: ---------|-----------------------------------------X------|------------------
q5: ---------|------------------------------------------------X------------------
q6: ---------X--------------------H----------------------------------@------@----
q7: -----------------------------------------------------------------X------|----
q8: ------------------------------------------------------------------------X----


### Fixed Noisy-Simulation Parameters

For this first QEC example, the hidden-memory simulation uses the requested fixed values `\Delta = 1`, `\sigma = 0.2`, and `N_{MC} = 100`. The current blockwise workflow still uses `window_size = 2`.

The notebook now also exposes `NUM_WORKERS` for Monte Carlo trajectory parallelism. The current implementation parallelizes over independent shot chunks using worker-local prepared state.

Because the Shor encoding depth is 11, the final fixed-window partition contains five 2-segment blocks and one trailing 1-segment block.


In [4]:
shor_model = SharedQuasiStaticModel(
    num_qubits=shor_layout.total_qubits,
    sigma2=SIGMA ** 2,
    segment_durations=(DELTA,) * shor_circuit.circuit_depth,
)
prepared_shor = prepare_blockwise_hidden_memory_simulation(
    shor_model,
    shor_circuit,
    window_size=WINDOW_SIZE,
)

print('Simulation configuration')
print('  window_size =', WINDOW_SIZE)
print('  Delta =', DELTA)
print('  sigma =', SIGMA)
print('  sigma^2 =', shor_model.sigma2)
print('  N_MC =', N_MC)
print('  NUM_WORKERS =', NUM_WORKERS)
print('  num_blocks =', len(prepared_shor.blocks))
print('  block spans =', [(block.start_segment, block.end_segment) for block in prepared_shor.blocks])


Simulation configuration
  window_size = 2
  Delta = 1.0
  sigma = 0.2
  sigma^2 = 0.04000000000000001
  N_MC = 100
  NUM_WORKERS = 4
  num_blocks = 6
  block spans = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9), (10, 10)]


### Monte Carlo Noisy Simulation

This cell runs the Shor Monte Carlo batch through the process-based `num_workers` API. The second cell below benchmarks the same `N_{MC}` in serial and parallel modes using the same random seed so the comparison is directly operational rather than analytic.


In [5]:
shor_batch, elapsed = run_mc_batch(
    shor_model,
    shor_circuit,
    window_size=WINDOW_SIZE,
    num_shots=N_MC,
    seed=SEED,
    num_workers=NUM_WORKERS,
)
shor_summary = summarize_batch(shor_batch)

print('Shor noisy-simulation summary')
print('  elapsed_seconds =', elapsed)
print('  num_shots =', shor_summary['num_shots'])
print('  num_workers =', NUM_WORKERS)
print('  mean_xi =', shor_summary['mean_xi'])
print('  std_xi =', shor_summary['std_xi'])
print('  mean_xi^2 =', shor_summary['mean_xi2'])
print('  avg_nontrivial_physical_block_faults =', shor_summary['avg_nontrivial_blocks'])
print('')
print_frequency_table('Physical sampled block frequencies', shor_summary['physical_frequencies'])
print('')
print_frequency_table('Toggling-frame sampled block frequencies', shor_summary['toggling_frequencies'])


Shor noisy-simulation summary
  elapsed_seconds = 165.936701226
  num_shots = 100
  num_workers = 4
  mean_xi = 0.013198914228604443
  std_xi = 0.17816904641116604
  mean_xi^2 = 0.0319184204358783
  avg_nontrivial_physical_block_faults = 2.42

Physical sampled block frequencies
  block 0: {'I': 0.53, 'ZIIIIIIII': 0.05, 'IIIIIIIIZ': 0.04, 'IIIIIZIII': 0.04, 'IIIZIIIII': 0.04, 'IIIIIIIZI': 0.03, 'IZIIIIIII': 0.02, 'IIIIIZIZZ': 0.01, 'IIIIZIIII': 0.01, 'IIIIZIIIZ': 0.01, 'IIIIZIIZI': 0.01, 'IIIIZZIIZ': 0.01, 'IIIZIIIZZ': 0.01, 'IIIZZIIIZ': 0.01, 'IIIZZZIII': 0.01, 'IIZIIIIII': 0.01, 'IIZIZIIII': 0.01, 'IIZZIIIIZ': 0.01, 'IIZZZIZZI': 0.01, 'IIZZZZZZI': 0.01, 'IZIIIIIZI': 0.01, 'IZIIZIIII': 0.01, 'IZIZIIZZI': 0.01, 'IZIZZZIII': 0.01, 'IZIZZZIZI': 0.01, 'IZZZZZIZZ': 0.01, 'ZIIIIIIZI': 0.01, 'ZIIIIIZZZ': 0.01, 'ZIIIZZZZZ': 0.01, 'ZIIZIIIZZ': 0.01, 'ZIZZZIZZI': 0.01, 'ZZIIZIIZI': 0.01}
  block 1: {'I': 0.58, 'ZIIZIIXII': 0.03, 'IIIIIIIIZ': 0.02, 'IIIIIZIII': 0.02, 'IIIIIZIIZ': 0.02, 'IIIIZIIII

### Serial Versus Parallel Timing

This timing probe uses the same Shor circuit, the same `N_{MC}`, and the same RNG seed in both modes. The sampled trajectories will not appear in the same order across modes, but the elapsed-time comparison is the quantity of interest here.


In [6]:
serial_batch, serial_elapsed = run_mc_batch(
    shor_model,
    shor_circuit,
    window_size=WINDOW_SIZE,
    num_shots=N_MC,
    seed=SEED,
    num_workers=1,
)

speedup = serial_elapsed / elapsed if elapsed > 0.0 else float('inf')
print('Timing comparison for the Shor example')
print('  serial_elapsed_seconds =', serial_elapsed)
print('  parallel_elapsed_seconds =', elapsed)
print('  num_workers =', NUM_WORKERS)
print('  speedup =', speedup)
print('  serial_num_shots =', serial_batch.num_shots)
print('  parallel_num_shots =', shor_batch.num_shots)


Timing comparison for the Shor example
  serial_elapsed_seconds = 531.943520146
  parallel_elapsed_seconds = 165.936701226
  num_workers = 4
  speedup = 3.2057014284110146
  serial_num_shots = 100
  parallel_num_shots = 100


### Block-Level Inspection

The Monte Carlo batch above uses the parallel worker path. The block channels remain conditional distributions in the toggling frame, while the sampled physical frequencies include the block-end conjugation step. The tables below inspect one representative shot and the prepared-object profiling from a separate single-shot prepared run.


In [7]:
inspection_process = build_prepared_blockwise_hidden_memory_process(
    prepared_shor,
    rng=np.random.default_rng(SEED),
)
print('Representative single-shot inspection: xi =', inspection_process.hidden_memory.xi)
print('')
for block_index, block_channel in enumerate(inspection_process.block_channels):
    print(f'Block {block_index} top toggling-frame probabilities =')
    print(dict(sort_probability_dict(block_channel.probabilities, top_k=8)))
    sampled_fault = inspection_process.block_faults[block_index]
    print('  sampled Q_b =', sampled_fault.toggling_frame_pauli_string)
    print('  sampled F_b =', sampled_fault.physical_pauli_string)
    print('')

print('Prepared simulation profiling')
print('  max candidate support size by block =', prepared_shor.block_candidate_support_sizes)
print('  cumulative exact-compute seconds by block =', prepared_shor.block_compute_seconds)


Representative single-shot inspection: xi = 0.02514604421867866

Block 0 top toggling-frame probabilities =
{'IIIIIIIII': 0.9787219724333064, 'ZIIIIIIII': 0.002480047549690683, 'ZIIZIIIII': 0.0024796558966913755, 'IZIIIIIII': 0.0024796558966913734, 'IIZIIIIII': 0.002479655896691372, 'IIIIZIIII': 0.002479655896691371, 'IIIIIIIIZ': 0.0024796558966913708, 'IIIIIIIZI': 0.0024796558966913708}
  sampled Q_b = None
  sampled F_b = None

Block 1 top toggling-frame probabilities =
{'IIIIIIIII': 0.9787219714413684, 'XIIIIIIII': 0.0024796558941782357, 'XIIIIIZII': 0.0024796558941782357, 'IZIIIIIII': 0.0024796558941782335, 'IIZIIIIII': 0.002479655894178232, 'IIIIZIIII': 0.0024796558941782313, 'IIIIIIIIZ': 0.002479655894178231, 'IIIIIIIZI': 0.002479655894178231}
  sampled Q_b = None
  sampled F_b = None

Block 2 top toggling-frame probabilities =
{'IIIIIIIII': 0.9824473484880887, 'IIZIIIIII': 0.0024890943796945235, 'IIIIZIIII': 0.002489094379694523, 'IIIIIIIIZ': 0.002489094379694522, 'IIIIIIIZI': 0

## Distance-2 Rotated Surface Code

This second example uses a 7-qubit distance-2 rotated surface-code syndrome-extraction round. The circuit is built from 4 data qubits and 3 ancilla qubits, matching the smallest rotated-memory patch size supported by Stim. As in the Shor section, the noisy simulation uses the same fixed `\Delta`, `\sigma`, `N_{MC}`, and `NUM_WORKERS` settings.


In [8]:
surface_layout = build_rotated_surface_d2_layout()
surface_circuit = build_rotated_surface_d2_syndrome_circuit()

print('Distance-2 rotated surface-code layout')
print('  total_qubits =', surface_layout.total_qubits)
print('  data_qubits =', surface_layout.data_qubits)
print('  x_ancilla =', surface_layout.x_ancilla)
print('  z_ancillas =', surface_layout.z_ancillas)
print('  x_stabilizer_support =', surface_layout.x_stabilizer_support)
print('  z_stabilizer_supports =', surface_layout.z_stabilizer_supports)
print('  circuit_depth =', surface_circuit.circuit_depth)
print('')
print('ASCII circuit:')
print(plot_rotated_surface_d2_syndrome_circuit())


Distance-2 rotated surface-code layout
  total_qubits = 7
  data_qubits = (0, 1, 2, 3)
  x_ancilla = 4
  z_ancillas = (5, 6)
  x_stabilizer_support = (0, 1, 2, 3)
  z_stabilizer_supports = ((0, 1), (2, 3))
  circuit_depth = 10

ASCII circuit:
q0: ---------X---------------------------@--------------------------------
q1: ---------|------X--------------------|------@-------------------------
q2: ---------|------|------X-------------|------|------@------------------
q3: ---------|------|------|------X------|------|------|------@-----------
q4: --H------@------@------@------@------|------|------|------|------H----
q5: -------------------------------------X------X------|------|-----------
q6: ---------------------------------------------------X------X-----------


### Surface-Code Noisy Simulation

This run uses the same blockwise hidden-memory workflow as the Shor example, but on the smaller 7-qubit rotated-patch control schedule. Because the circuit is shorter and the support is smaller, this section is expected to finish much faster than the Shor section.


In [9]:
surface_model = SharedQuasiStaticModel(
    num_qubits=surface_layout.total_qubits,
    sigma2=SIGMA ** 2,
    segment_durations=(DELTA,) * surface_circuit.circuit_depth,
)
prepared_surface = prepare_blockwise_hidden_memory_simulation(
    surface_model,
    surface_circuit,
    window_size=WINDOW_SIZE,
)
surface_batch, surface_elapsed = run_mc_batch(
    surface_model,
    surface_circuit,
    window_size=WINDOW_SIZE,
    num_shots=N_MC,
    seed=SEED,
    num_workers=NUM_WORKERS,
)
surface_summary = summarize_batch(surface_batch)

print('Distance-2 rotated surface-code noisy-simulation summary')
print('  elapsed_seconds =', surface_elapsed)
print('  num_shots =', surface_summary['num_shots'])
print('  num_workers =', NUM_WORKERS)
print('  mean_xi =', surface_summary['mean_xi'])
print('  std_xi =', surface_summary['std_xi'])
print('  mean_xi^2 =', surface_summary['mean_xi2'])
print('  avg_nontrivial_physical_block_faults =', surface_summary['avg_nontrivial_blocks'])
print('')
print_frequency_table('Physical sampled block frequencies', surface_summary['physical_frequencies'])
print('')
print_frequency_table('Toggling-frame sampled block frequencies', surface_summary['toggling_frequencies'])


Distance-2 rotated surface-code noisy-simulation summary
  elapsed_seconds = 9.596237085999974
  num_shots = 100
  num_workers = 4
  mean_xi = 0.011826147435162027
  std_xi = 0.2204297928402039
  mean_xi^2 = 0.04872915133473339
  avg_nontrivial_physical_block_faults = 2.18

Physical sampled block frequencies
  block 0: {'I': 0.59, 'IZIIIII': 0.04, 'IIIIIIZ': 0.03, 'IIIZIII': 0.03, 'IIIIIZI': 0.02, 'IIZIIII': 0.02, 'IIZIIZI': 0.02, 'IIIIIZZ': 0.01, 'IIIZIZI': 0.01, 'IIZIIIZ': 0.01, 'IIZIZII': 0.01, 'IIZZIIZ': 0.01, 'IIZZIZI': 0.01, 'IZIIIIZ': 0.01, 'IZIIZII': 0.01, 'IZIIZIZ': 0.01, 'IZIZZZI': 0.01, 'IZZIIII': 0.01, 'IZZIZZI': 0.01, 'IZZZIIZ': 0.01, 'XIIIIIZ': 0.01, 'XIIIZZI': 0.01, 'XIIZZII': 0.01, 'XIZIXZI': 0.01, 'XZIIZZI': 0.01, 'YIIIYII': 0.01, 'YZIIYZI': 0.01, 'YZZZYII': 0.01, 'YZZZZZZ': 0.01, 'ZIIIZIZ': 0.01, 'ZIZZIZI': 0.01, 'ZZZZZZZ': 0.01}
  block 1: {'I': 0.53, 'IIIZIII': 0.04, 'IIIIIIZ': 0.03, 'XXXIZII': 0.03, 'XYXIYZI': 0.03, 'IIIIIZI': 0.02, 'XXYIYII': 0.02, 'XYXIYII': 0.02

### Surface-Code Block Inspection

This inspection cell shows one representative sampled shot together with the prepared-object profiling for the smaller rotated-patch example.


In [10]:
surface_inspection_process = build_prepared_blockwise_hidden_memory_process(
    prepared_surface,
    rng=np.random.default_rng(SEED),
)
print('Representative surface-code single-shot inspection: xi =', surface_inspection_process.hidden_memory.xi)
print('')
for block_index, block_channel in enumerate(surface_inspection_process.block_channels):
    print(f'Block {block_index} top toggling-frame probabilities =')
    print(dict(sort_probability_dict(block_channel.probabilities, top_k=8)))
    sampled_fault = surface_inspection_process.block_faults[block_index]
    print('  sampled Q_b =', sampled_fault.toggling_frame_pauli_string)
    print('  sampled F_b =', sampled_fault.physical_pauli_string)
    print('')

print('Prepared surface-code profiling')
print('  max candidate support size by block =', prepared_surface.block_candidate_support_sizes)
print('  cumulative exact-compute seconds by block =', prepared_surface.block_compute_seconds)


Representative surface-code single-shot inspection: xi = 0.02514604421867866

Block 0 top toggling-frame probabilities =
{'IIIIIII': 0.9849332897337957, 'IIIZIII': 0.0024953926738397005, 'IIIIIIZ': 0.0024953926738397, 'IIIIIZI': 0.0024953926738397, 'IIZIIII': 0.0024953926738397, 'IZIIIII': 0.0024953926738396992, 'IIIIXII': 0.0006230591376416768, 'XIIIXII': 0.0006230591376416768}
  sampled Q_b = None
  sampled F_b = None

Block 1 top toggling-frame probabilities =
{'IIIIIII': 0.9849332897337957, 'IIIZIII': 0.0024953926738397005, 'IIIIIIZ': 0.0024953926738397, 'IIIIIZI': 0.0024953926738397, 'IZIIZII': 0.0024953926738397, 'ZIIIZII': 0.0024953926738396992, 'IIZIIII': 0.0006230591376416768, 'IIZIZII': 0.0006230591376416768}
  sampled Q_b = None
  sampled F_b = None

Block 2 top toggling-frame probabilities =
{'IIIIIII': 0.9849332897337957, 'IIZIZII': 0.0024953926738397005, 'IIIIIIZ': 0.0024953926738397, 'IIIZZII': 0.0024953926738397, 'IZIIZII': 0.0024953926738397, 'ZIIIZII': 0.0024953926738

## Distance-3 Rotated Memory-X Surface Code

This third example uses the standard 17-qubit rotated distance-3 patch: 9 data qubits plus 8 ancilla qubits. The stabilizer supports match the usual rotated surface-17 layout associated with Stim's `surface_code:rotated_memory_x` family.

Because the current `lichen` circuit model treats each ideal gate as its own noisy segment, this example is much heavier than the Shor and 7-qubit surface-code sections. The cells below are provided in the same format, but they are intentionally left unexecuted in this update.


In [ ]:
surface_d3_layout = build_rotated_surface_d3_memory_x_layout()
surface_d3_circuit = build_rotated_surface_d3_memory_x_circuit()

print('Distance-3 rotated-memory-X surface-code layout')
print('  total_qubits =', surface_d3_layout.total_qubits)
print('  data_qubits =', surface_d3_layout.data_qubits)
print('  x_ancillas =', surface_d3_layout.x_ancillas)
print('  z_ancillas =', surface_d3_layout.z_ancillas)
print('  x_stabilizer_supports =', surface_d3_layout.x_stabilizer_supports)
print('  z_stabilizer_supports =', surface_d3_layout.z_stabilizer_supports)
print('  circuit_depth =', surface_d3_circuit.circuit_depth)
print('')
print('ASCII circuit:')
print(plot_rotated_surface_d3_memory_x_circuit())


### Heavy Exact Simulation Scaffold

These cells use the same fixed `\Delta`, `\sigma`, `N_{MC}`, and `NUM_WORKERS` pattern as the earlier examples. In practice, the exact d=3 run is currently a heavy demonstration workload because the circuit has 32 noisy segments in the present one-gate-per-layer model.


In [ ]:
surface_d3_model = SharedQuasiStaticModel(
    num_qubits=surface_d3_layout.total_qubits,
    sigma2=SIGMA ** 2,
    segment_durations=(DELTA,) * surface_d3_circuit.circuit_depth,
)
prepared_surface_d3 = prepare_blockwise_hidden_memory_simulation(
    surface_d3_model,
    surface_d3_circuit,
    window_size=WINDOW_SIZE,
)
surface_d3_batch, surface_d3_elapsed = run_mc_batch(
    surface_d3_model,
    surface_d3_circuit,
    window_size=WINDOW_SIZE,
    num_shots=N_MC,
    seed=SEED,
    num_workers=NUM_WORKERS,
)
surface_d3_summary = summarize_batch(surface_d3_batch)

print('Distance-3 rotated-memory-X noisy-simulation summary')
print('  elapsed_seconds =', surface_d3_elapsed)
print('  num_shots =', surface_d3_summary['num_shots'])
print('  num_workers =', NUM_WORKERS)
print('  mean_xi =', surface_d3_summary['mean_xi'])
print('  std_xi =', surface_d3_summary['std_xi'])
print('  mean_xi^2 =', surface_d3_summary['mean_xi2'])
print('  avg_nontrivial_physical_block_faults =', surface_d3_summary['avg_nontrivial_blocks'])
print('')
print_frequency_table('Physical sampled block frequencies', surface_d3_summary['physical_frequencies'])
print('')
print_frequency_table('Toggling-frame sampled block frequencies', surface_d3_summary['toggling_frequencies'])


### Distance-3 Block Inspection

This inspection cell mirrors the earlier examples. It is also left unexecuted here because even a single exact d=3 shot is already a large workload in the current implementation.


In [ ]:
surface_d3_inspection_process = build_prepared_blockwise_hidden_memory_process(
    prepared_surface_d3,
    rng=np.random.default_rng(SEED),
)
print('Representative d=3 single-shot inspection: xi =', surface_d3_inspection_process.hidden_memory.xi)
print('')
for block_index, block_channel in enumerate(surface_d3_inspection_process.block_channels):
    print(f'Block {block_index} top toggling-frame probabilities =')
    print(dict(sort_probability_dict(block_channel.probabilities, top_k=8)))
    sampled_fault = surface_d3_inspection_process.block_faults[block_index]
    print('  sampled Q_b =', sampled_fault.toggling_frame_pauli_string)
    print('  sampled F_b =', sampled_fault.physical_pauli_string)
    print('')

print('Prepared d=3 profiling')
print('  max candidate support size by block =', prepared_surface_d3.block_candidate_support_sizes)
print('  cumulative exact-compute seconds by block =', prepared_surface_d3.block_compute_seconds)


## Future QECC Sections

Add later QECC circuit sections below this point. The intended pattern is:

1. load or construct a circuit from `examples/QEC_examples/`
2. print the layout / visualization
3. fix the simulation parameters
4. run the prepared noisy simulation
5. summarize the physical and toggling-frame behavior
